# Cityscapes: полный запуск эксперимента в Google Colab

Notebook только настраивает окружение и вызывает скрипты проекта. Dataset, обучение, метрики и оценка остаются в `src/`. Cityscapes загружается через `kagglehub.dataset_download("electraawais/cityscape-dataset")`. Перед запуском выберите **Runtime → Change runtime type → GPU**.

## 1. Проверка GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU не найден. Выберите Runtime → Change runtime type → GPU.")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
!nvidia-smi

## 2. Подключение Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import kagglehub

path = kagglehub.dataset_download("electraawais/cityscape-dataset")
print('KaggleHub dataset:', path)

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/atikkhon/diploma.git"
CITYSCAPES_ROOT = Path(path).expanduser().resolve()

PROJECT_DIR = Path('/content/Diploma')
DRIVE_ROOT = Path('/content/drive/MyDrive/cityscapes_robustness')
RUNS_DIR = DRIVE_ROOT / 'runs'
MODELS_DIR = DRIVE_ROOT / 'models'
LOG_DIR = DRIVE_ROOT / 'logs'
MLFLOW_DB = DRIVE_ROOT / 'mlflow.db'
MLFLOW_ARTIFACT_DIR = DRIVE_ROOT / 'mlartifacts'
SPLIT_MANIFEST = DRIVE_ROOT / 'split_manifest.csv'
RUNTIME_CONFIG = DRIVE_ROOT / 'experiment_colab.yaml'

for directory in (DRIVE_ROOT, RUNS_DIR, MODELS_DIR, LOG_DIR, MLFLOW_ARTIFACT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

os.environ.update({
    'REPO_URL': REPO_URL,
    'PROJECT_DIR': str(PROJECT_DIR),
    'DRIVE_ROOT': str(DRIVE_ROOT),
    'RUNS_DIR': str(RUNS_DIR),
    'MODELS_DIR': str(MODELS_DIR),
    'LOG_DIR': str(LOG_DIR),
    'SPLIT_MANIFEST': str(SPLIT_MANIFEST),
    'RUNTIME_CONFIG': str(RUNTIME_CONFIG),
    'MLFLOW_TRACKING_URI': f'sqlite:///{MLFLOW_DB}',
    'MLFLOW_ARTIFACT_ROOT': MLFLOW_ARTIFACT_DIR.as_uri(),
})
print('Drive root:', DRIVE_ROOT)
print('Runs:', RUNS_DIR)
print('Models:', MODELS_DIR)
print('MLflow tracking URI:', os.environ['MLFLOW_TRACKING_URI'])
print('MLflow artifact root:', os.environ['MLFLOW_ARTIFACT_ROOT'])


## 3. Клонирование или обновление Git-репозитория

При повторном запуске выполняется `git pull --ff-only`; локальные изменения в `/content/Diploma` не перезаписываются.

In [ ]:
%%bash
set -euo pipefail
if [[ -d "$PROJECT_DIR/.git" ]]; then
  git -C "$PROJECT_DIR" pull --ff-only 2>&1 | tee -a "$LOG_DIR/git_update.log"
else
  git clone "$REPO_URL" "$PROJECT_DIR" 2>&1 | tee -a "$LOG_DIR/git_update.log"
fi

## 4. Установка `requirements.txt`

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -m pip install -r requirements.txt 2>&1 | tee -a "$LOG_DIR/install.log"

### Проверка SQLite MLflow

In [ ]:
import sys
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from src.tracking import check_mlflow_connection

mlflow_info = check_mlflow_connection('cityscapes_segmentation')
print(mlflow_info)


## 5. Пути Cityscapes и Colab-конфигурация

Manifest, checkpoint, история обучения, логи и MLflow переживают разрыв runtime, потому что сохраняются в Google Drive.

In [ ]:
import sys
import yaml

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from src.dataset import discover_cityscapes_layout, prepare_train_id_masks

layout = discover_cityscapes_layout(CITYSCAPES_ROOT)
# Маски читаются на каждой эпохе, поэтому держим кэш на локальном SSD Colab.
# После обрыва runtime эта ячейка быстро восстановит его из Kaggle labelIds.
prepared_gtfine = Path('/content/prepared_gtFine')
train_masks = prepare_train_id_masks(
    layout['train_masks'], prepared_gtfine / 'train'
)
val_masks = prepare_train_id_masks(
    layout['val_masks'], prepared_gtfine / 'val'
)

with (PROJECT_DIR / 'configs/experiment.yaml').open(encoding='utf-8') as file:
    config = yaml.safe_load(file)
config['data']['root'] = str(CITYSCAPES_ROOT)
config['data']['train_images'] = str(layout['train_images'])
config['data']['train_masks'] = str(train_masks)
config['data']['official_val_images'] = str(layout['val_images'])
config['data']['official_val_masks'] = str(val_masks)
config['data']['split_file'] = str(SPLIT_MANIFEST)
config['training']['log_interval'] = 25
with RUNTIME_CONFIG.open('w', encoding='utf-8') as file:
    yaml.safe_dump(config, file, allow_unicode=True, sort_keys=False)
print('Runtime config:', RUNTIME_CONFIG)
print('Train images:', layout['train_images'])
print('Train masks:', train_masks)
print('Official val images:', layout['val_images'])
print('Official val masks:', val_masks)


## 6. Создание `split_manifest.csv`

Manifest пересоздаётся детерминированно, чтобы в нём всегда были актуальные локальные пути. При повторном запуске полная проверка масок не дублируется.

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
if [[ -s "$SPLIT_MANIFEST" ]]; then
  echo "Обновление путей существующего manifest без повторной полной проверки масок" | tee -a "$LOG_DIR/create_split.log"
  python scripts/create_split.py --config "$RUNTIME_CONFIG" --skip-mask-validation 2>&1 | tee -a "$LOG_DIR/create_split.log"
else
  python scripts/create_split.py --config "$RUNTIME_CONFIG" 2>&1 | tee -a "$LOG_DIR/create_split.log"
fi

## 7. Запуск тестов

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -m pytest tests/test_pipeline.py -v 2>&1 | tee -a "$LOG_DIR/pytest.log"

## 8. Dataset smoke test

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/smoke_test_dataset.py --config "$RUNTIME_CONFIG" --split dev 2>&1 | tee -a "$LOG_DIR/dataset_smoke_test.log"

In [ ]:
from IPython.display import Image, display
smoke_png = PROJECT_DIR / 'outputs/figures/dataset_smoke_test.png'
if not smoke_png.is_file():
    raise FileNotFoundError(f'Smoke-test PNG не найден: {smoke_png}')
display(Image(filename=str(smoke_png)))

## Выбор отдельной модульной версии

Первые восемь разделов используют стабильную `main`. Теперь переключите локальный Colab-клон на отдельную ветку U-Net и проверьте её тестами.

In [ ]:
%%bash
set -euo pipefail
git -C "$PROJECT_DIR" fetch origin codex/modular-segmentation-pipeline
git -C "$PROJECT_DIR" switch codex/modular-segmentation-pipeline
git -C "$PROJECT_DIR" pull --ff-only
cd "$PROJECT_DIR"
python -m pytest tests/test_pipeline.py -q

## Model run helper

Run this helper once after section 8. Each model section below calls it with its own settings.


In [ ]:
import os
import yaml
from IPython.display import Image, display


CORRUPTION_CONFIG = {
    'darkness': {
        'levels': {
            1: {'factor': 0.75},
            2: {'factor': 0.55},
            3: {'factor': 0.35},
        }
    },
    'brightness': {
        'levels': {
            1: {'factor': 1.15},
            2: {'factor': 1.35},
            3: {'factor': 1.60},
        }
    },
    'gaussian_blur': {
        'levels': {
            1: {'kernel_size': 5, 'sigma': 1.0},
            2: {'kernel_size': 9, 'sigma': 2.0},
            3: {'kernel_size': 13, 'sigma': 3.0},
        }
    },
    'gaussian_noise': {
        'levels': {
            1: {'sigma': 8.0},
            2: {'sigma': 16.0},
            3: {'sigma': 28.0},
        }
    },
    'jpeg_compression': {
        'levels': {
            1: {'quality': 70},
            2: {'quality': 40},
            3: {'quality': 15},
        }
    },
    'fog': {
        'levels': {
            1: {'alpha': 0.15},
            2: {'alpha': 0.30},
            3: {'alpha': 0.45},
        }
    },
}


def prepare_model_run(
    selected_model,
    run_name,
    resume_training,
    continue_from_run,
    init_checkpoint,
    model_settings,
    update_corruption_settings=False,
):
    if resume_training and continue_from_run is not None:
        raise ValueError('Choose only one mode: resume_training or continue_from_run')
    if init_checkpoint not in {'last', 'best'}:
        raise ValueError("init_checkpoint must be 'last' or 'best'")

    run_dir = DRIVE_ROOT / 'runs' / run_name
    model_dir = DRIVE_ROOT / 'models' / run_name
    run_config_path = run_dir / 'run_config.yaml'
    run_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)

    if resume_training and run_config_path.exists():
        with run_config_path.open(encoding='utf-8') as file:
            run_config = yaml.safe_load(file)

        config_updated = False
        if update_corruption_settings:
            run_config['corruptions'] = CORRUPTION_CONFIG
            config_updated = True
            print('Replaced corruption settings from CORRUPTION_CONFIG')
        else:
            if 'corruptions' not in run_config:
                run_config['corruptions'] = {}
            added_corruptions = []
            for corruption_name, corruption_settings in CORRUPTION_CONFIG.items():
                if corruption_name not in run_config['corruptions']:
                    run_config['corruptions'][corruption_name] = corruption_settings
                    added_corruptions.append(corruption_name)
            if added_corruptions:
                config_updated = True
                print('Added missing corruption settings:', ', '.join(added_corruptions))

        if config_updated:
            with run_config_path.open('w', encoding='utf-8') as file:
                yaml.safe_dump(run_config, file, allow_unicode=True, sort_keys=False)
        print('Resume uses the existing run_config.yaml without overwriting training settings')
    else:
        with RUNTIME_CONFIG.open(encoding='utf-8') as file:
            run_config = yaml.safe_load(file)
        run_config['project_root'] = str(PROJECT_DIR)
        run_config['run'] = {
            'name': run_name,
            'output_dir': str(run_dir),
            'model_dir': str(model_dir),
        }
        run_config['model'] = model_settings['model']
        run_config['training'].update(model_settings['training'])
        run_config['evaluation'].update(model_settings['evaluation'])
        run_config['tracking']['experiment_name'] = 'cityscapes_segmentation'
        for key in ('checkpoint_dir', 'history_dir'):
            run_config['training'].pop(key, None)
        for key in ('metrics_dir', 'figures_dir', 'predictions_dir'):
            run_config['evaluation'].pop(key, None)
        run_config['corruptions'] = CORRUPTION_CONFIG
        with run_config_path.open('w', encoding='utf-8') as file:
            yaml.safe_dump(run_config, file, allow_unicode=True, sort_keys=False)

    os.environ['RUN_CONFIG'] = str(run_config_path)
    os.environ['RUN_NAME'] = run_name
    os.environ['RUN_DIR'] = str(run_dir)
    os.environ['MODEL_DIR'] = str(model_dir)
    if 'IMAGE_INDEX' not in globals():
        globals()['IMAGE_INDEX'] = 0
    os.environ['IMAGE_INDEX'] = str(globals()['IMAGE_INDEX'])
    os.environ['RESUME_TRAINING'] = '1' if resume_training else '0'
    os.environ['CONTINUE_FROM_RUN'] = '' if continue_from_run is None else str(continue_from_run)
    os.environ['INIT_CHECKPOINT'] = init_checkpoint

    globals().update(
        {
            'SELECTED_MODEL': selected_model,
            'RUN_NAME': run_name,
            'RUN_DIR': run_dir,
            'MODEL_DIR': model_dir,
            'RUN_CONFIG': run_config_path,
            'RESUME_TRAINING': resume_training,
            'CONTINUE_FROM_RUN': continue_from_run,
            'INIT_CHECKPOINT': init_checkpoint,
            'UPDATE_CORRUPTION_SETTINGS': update_corruption_settings,
        }
    )

    print('Model:', selected_model)
    print('Run:', run_name)
    print('Results dir:', run_dir)
    print('Model dir:', model_dir)
    print('Config:', run_config_path)
    print('Resume:', resume_training)
    print('Continue from run:', continue_from_run)
    print('Init checkpoint:', init_checkpoint)
    print('Update corruption settings:', update_corruption_settings)
    print('Configured epochs:', run_config['training']['epochs'])


## U-Net

Collapse this section when you work with another model.


### U-Net: parameters


In [ ]:
UNET_SETTINGS = {
    'model': {
        'name': 'unet',
        'encoder_name': 'resnet34',
        'encoder_weights': 'imagenet',
    },
    'training': {
        'epochs': 8,
        'batch_size': 8,
        'num_workers': 2,
        'log_interval': 25,
        'learning_rate': 0.0003,
        'weight_decay': 0.0001,
        'device': 'auto',
        'mixed_precision': True,
    },
    'evaluation': {
        'batch_size': 8,
    },
}

prepare_model_run(
    selected_model='unet',
    run_name='unet_experiment_01',
    resume_training=False,
    continue_from_run=None,
    init_checkpoint='last',
    model_settings=UNET_SETTINGS,
    update_corruption_settings=False,
)


### U-Net: train, resume, or continue


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
if [[ -n "$CONTINUE_FROM_RUN" ]]; then
  python -u scripts/train_model.py --config "$RUN_CONFIG" --continue-from-run "$CONTINUE_FROM_RUN" --init-checkpoint "$INIT_CHECKPOINT" 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
elif [[ "$RESUME_TRAINING" == "1" ]]; then
  python -u scripts/train_model.py --config "$RUN_CONFIG" --resume 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
else
  python -u scripts/train_model.py --config "$RUN_CONFIG" 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
fi


In [ ]:
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


### U-Net: preview image index

Set one official validation image index for all manual preview cells below.


In [ ]:
IMAGE_INDEX = 0
os.environ['IMAGE_INDEX'] = str(IMAGE_INDEX)
print('Preview image index:', IMAGE_INDEX)


### U-Net: clean segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition clean 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_clean.log"


In [ ]:
clean_preview = RUN_DIR / 'figures' / f'segmentation_clean_index_{IMAGE_INDEX}.png'
display(Image(filename=str(clean_preview)))


### U-Net: clean evaluation


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition clean 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_clean.log"


### U-Net: Darkness evaluation


In [ ]:
DARKNESS_SEVERITY = 1
if DARKNESS_SEVERITY not in {1, 2, 3}:
    raise ValueError('DARKNESS_SEVERITY must be 1, 2, or 3')
os.environ['DARKNESS_SEVERITY'] = str(DARKNESS_SEVERITY)
print('Darkness severity:', DARKNESS_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition darkness --severity "$DARKNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_darkness_s${DARKNESS_SEVERITY}.log"


### U-Net: Darkness segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition darkness --severity "$DARKNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_darkness_s${DARKNESS_SEVERITY}.log"


In [ ]:
darkness_preview = RUN_DIR / 'figures' / f'segmentation_darkness_s{DARKNESS_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(darkness_preview)))


### U-Net: Increased brightness evaluation


In [ ]:
BRIGHTNESS_SEVERITY = 1
if BRIGHTNESS_SEVERITY not in {1, 2, 3}:
    raise ValueError('BRIGHTNESS_SEVERITY must be 1, 2, or 3')
os.environ['BRIGHTNESS_SEVERITY'] = str(BRIGHTNESS_SEVERITY)
print('Increased brightness severity:', BRIGHTNESS_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition brightness --severity "$BRIGHTNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_brightness_s${BRIGHTNESS_SEVERITY}.log"


### U-Net: Increased brightness segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition brightness --severity "$BRIGHTNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_brightness_s${BRIGHTNESS_SEVERITY}.log"


In [ ]:
brightness_preview = RUN_DIR / 'figures' / f'segmentation_brightness_s{BRIGHTNESS_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(brightness_preview)))


### U-Net: Gaussian blur evaluation


In [ ]:
GAUSSIAN_BLUR_SEVERITY = 1
if GAUSSIAN_BLUR_SEVERITY not in {1, 2, 3}:
    raise ValueError('GAUSSIAN_BLUR_SEVERITY must be 1, 2, or 3')
os.environ['GAUSSIAN_BLUR_SEVERITY'] = str(GAUSSIAN_BLUR_SEVERITY)
print('Gaussian blur severity:', GAUSSIAN_BLUR_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition gaussian_blur --severity "$GAUSSIAN_BLUR_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_gaussian_blur_s${GAUSSIAN_BLUR_SEVERITY}.log"


### U-Net: Gaussian blur segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition gaussian_blur --severity "$GAUSSIAN_BLUR_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_gaussian_blur_s${GAUSSIAN_BLUR_SEVERITY}.log"


In [ ]:
gaussian_blur_preview = RUN_DIR / 'figures' / f'segmentation_gaussian_blur_s{GAUSSIAN_BLUR_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(gaussian_blur_preview)))


### U-Net: Gaussian noise evaluation


In [ ]:
GAUSSIAN_NOISE_SEVERITY = 1
if GAUSSIAN_NOISE_SEVERITY not in {1, 2, 3}:
    raise ValueError('GAUSSIAN_NOISE_SEVERITY must be 1, 2, or 3')
os.environ['GAUSSIAN_NOISE_SEVERITY'] = str(GAUSSIAN_NOISE_SEVERITY)
print('Gaussian noise severity:', GAUSSIAN_NOISE_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition gaussian_noise --severity "$GAUSSIAN_NOISE_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_gaussian_noise_s${GAUSSIAN_NOISE_SEVERITY}.log"


### U-Net: Gaussian noise segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition gaussian_noise --severity "$GAUSSIAN_NOISE_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_gaussian_noise_s${GAUSSIAN_NOISE_SEVERITY}.log"


In [ ]:
gaussian_noise_preview = RUN_DIR / 'figures' / f'segmentation_gaussian_noise_s{GAUSSIAN_NOISE_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(gaussian_noise_preview)))


### U-Net: JPEG compression evaluation


In [ ]:
JPEG_COMPRESSION_SEVERITY = 1
if JPEG_COMPRESSION_SEVERITY not in {1, 2, 3}:
    raise ValueError('JPEG_COMPRESSION_SEVERITY must be 1, 2, or 3')
os.environ['JPEG_COMPRESSION_SEVERITY'] = str(JPEG_COMPRESSION_SEVERITY)
print('JPEG compression severity:', JPEG_COMPRESSION_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition jpeg_compression --severity "$JPEG_COMPRESSION_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_jpeg_compression_s${JPEG_COMPRESSION_SEVERITY}.log"


### U-Net: JPEG compression segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition jpeg_compression --severity "$JPEG_COMPRESSION_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_jpeg_compression_s${JPEG_COMPRESSION_SEVERITY}.log"


In [ ]:
jpeg_compression_preview = RUN_DIR / 'figures' / f'segmentation_jpeg_compression_s{JPEG_COMPRESSION_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(jpeg_compression_preview)))


### U-Net: Fog veil evaluation


In [ ]:
FOG_SEVERITY = 1
if FOG_SEVERITY not in {1, 2, 3}:
    raise ValueError('FOG_SEVERITY must be 1, 2, or 3')
os.environ['FOG_SEVERITY'] = str(FOG_SEVERITY)
print('Fog veil severity:', FOG_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition fog --severity "$FOG_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_fog_s${FOG_SEVERITY}.log"


### U-Net: Fog veil segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition fog --severity "$FOG_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_fog_s${FOG_SEVERITY}.log"


In [ ]:
fog_preview = RUN_DIR / 'figures' / f'segmentation_fog_s{FOG_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(fog_preview)))


### U-Net: batch preview selected image indices

Use this optional block after you have chosen several good validation image indices.


In [ ]:
PREVIEW_INDICES = [0, 1, 2]
if not PREVIEW_INDICES:
    raise ValueError('PREVIEW_INDICES must contain at least one image index')

BATCH_PREVIEW_SEVERITIES = {
    'darkness': globals().get('DARKNESS_SEVERITY', 1),
    'brightness': globals().get('BRIGHTNESS_SEVERITY', 1),
    'gaussian_blur': globals().get('GAUSSIAN_BLUR_SEVERITY', 1),
    'gaussian_noise': globals().get('GAUSSIAN_NOISE_SEVERITY', 1),
    'jpeg_compression': globals().get('JPEG_COMPRESSION_SEVERITY', 1),
    'fog': globals().get('FOG_SEVERITY', 1),
}
for condition, severity in BATCH_PREVIEW_SEVERITIES.items():
    if severity not in {1, 2, 3}:
        raise ValueError(f'{condition} severity must be 1, 2, or 3')

os.environ['PREVIEW_INDICES'] = ' '.join(str(index) for index in PREVIEW_INDICES)
os.environ['DARKNESS_SEVERITY'] = str(BATCH_PREVIEW_SEVERITIES['darkness'])
os.environ['BRIGHTNESS_SEVERITY'] = str(BATCH_PREVIEW_SEVERITIES['brightness'])
os.environ['GAUSSIAN_BLUR_SEVERITY'] = str(BATCH_PREVIEW_SEVERITIES['gaussian_blur'])
os.environ['GAUSSIAN_NOISE_SEVERITY'] = str(BATCH_PREVIEW_SEVERITIES['gaussian_noise'])
os.environ['JPEG_COMPRESSION_SEVERITY'] = str(BATCH_PREVIEW_SEVERITIES['jpeg_compression'])
os.environ['FOG_SEVERITY'] = str(BATCH_PREVIEW_SEVERITIES['fog'])

print('Batch preview indices:', PREVIEW_INDICES)
print('Batch preview severities:', BATCH_PREVIEW_SEVERITIES)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"

{
  for image_index in $PREVIEW_INDICES; do
    echo "Preview image index: ${image_index}"

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition clean

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition darkness --severity "$DARKNESS_SEVERITY"

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition brightness --severity "$BRIGHTNESS_SEVERITY"

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition gaussian_blur --severity "$GAUSSIAN_BLUR_SEVERITY"

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition gaussian_noise --severity "$GAUSSIAN_NOISE_SEVERITY"

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition jpeg_compression --severity "$JPEG_COMPRESSION_SEVERITY"

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition fog --severity "$FOG_SEVERITY"
  done
} 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_batch.log"


In [ ]:
batch_preview_files = []
for image_index in PREVIEW_INDICES:
    batch_preview_files.append(RUN_DIR / 'figures' / f'segmentation_clean_index_{image_index}.png')
    for condition, severity in BATCH_PREVIEW_SEVERITIES.items():
        batch_preview_files.append(
            RUN_DIR / 'figures' / f'segmentation_{condition}_s{severity}_index_{image_index}.png'
        )

for preview_path in batch_preview_files:
    print(preview_path.name)
    display(Image(filename=str(preview_path)))


### U-Net: saved results


In [ ]:
import pandas as pd

print('Results dir:', RUN_DIR)
print('Model dir:', MODEL_DIR)
display(pd.read_csv(RUN_DIR / 'metrics' / 'training_history.csv'))
display(pd.read_csv(RUN_DIR / 'metrics' / 'evaluation_results.csv'))

for plot_name in [
    'training_loss_curve.png',
    'dev_miou_curve.png',
    'dev_per_class_iou_curve.png',
]:
    plot_path = RUN_DIR / 'figures' / plot_name
    print(plot_path)
    display(Image(filename=str(plot_path)))


## DeepLabV3+

Collapse this section when you work with another model.


### DeepLabV3+: parameters


In [ ]:
DEEPLABV3PLUS_SETTINGS = {
    'model': {
        'name': 'deeplabv3plus',
        'encoder_name': 'resnet34',
        'encoder_weights': 'imagenet',
    },
    'training': {
        'epochs': 8,
        'batch_size': 8,
        'num_workers': 2,
        'log_interval': 25,
        'learning_rate': 0.0003,
        'weight_decay': 0.0001,
        'device': 'auto',
        'mixed_precision': True,
    },
    'evaluation': {
        'batch_size': 8,
    },
}

prepare_model_run(
    selected_model='deeplabv3plus',
    run_name='deeplabv3plus_experiment_01',
    resume_training=False,
    continue_from_run=None,
    init_checkpoint='last',
    model_settings=DEEPLABV3PLUS_SETTINGS,
    update_corruption_settings=False,
)


### DeepLabV3+: train, resume, or continue


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
if [[ -n "$CONTINUE_FROM_RUN" ]]; then
  python -u scripts/train_model.py --config "$RUN_CONFIG" --continue-from-run "$CONTINUE_FROM_RUN" --init-checkpoint "$INIT_CHECKPOINT" 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
elif [[ "$RESUME_TRAINING" == "1" ]]; then
  python -u scripts/train_model.py --config "$RUN_CONFIG" --resume 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
else
  python -u scripts/train_model.py --config "$RUN_CONFIG" 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
fi


In [ ]:
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


### DeepLabV3+: preview image index

Set one official validation image index for all manual preview cells below.


In [ ]:
IMAGE_INDEX = 0
os.environ['IMAGE_INDEX'] = str(IMAGE_INDEX)
print('Preview image index:', IMAGE_INDEX)


### DeepLabV3+: clean segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition clean 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_clean.log"


In [ ]:
clean_preview = RUN_DIR / 'figures' / f'segmentation_clean_index_{IMAGE_INDEX}.png'
display(Image(filename=str(clean_preview)))


### DeepLabV3+: clean evaluation


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition clean 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_clean.log"


### DeepLabV3+: Darkness evaluation


In [ ]:
DARKNESS_SEVERITY = 1
if DARKNESS_SEVERITY not in {1, 2, 3}:
    raise ValueError('DARKNESS_SEVERITY must be 1, 2, or 3')
os.environ['DARKNESS_SEVERITY'] = str(DARKNESS_SEVERITY)
print('Darkness severity:', DARKNESS_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition darkness --severity "$DARKNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_darkness_s${DARKNESS_SEVERITY}.log"


### DeepLabV3+: Darkness segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition darkness --severity "$DARKNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_darkness_s${DARKNESS_SEVERITY}.log"


In [ ]:
darkness_preview = RUN_DIR / 'figures' / f'segmentation_darkness_s{DARKNESS_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(darkness_preview)))


### DeepLabV3+: Increased brightness evaluation


In [ ]:
BRIGHTNESS_SEVERITY = 1
if BRIGHTNESS_SEVERITY not in {1, 2, 3}:
    raise ValueError('BRIGHTNESS_SEVERITY must be 1, 2, or 3')
os.environ['BRIGHTNESS_SEVERITY'] = str(BRIGHTNESS_SEVERITY)
print('Increased brightness severity:', BRIGHTNESS_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition brightness --severity "$BRIGHTNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_brightness_s${BRIGHTNESS_SEVERITY}.log"


### DeepLabV3+: Increased brightness segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition brightness --severity "$BRIGHTNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_brightness_s${BRIGHTNESS_SEVERITY}.log"


In [ ]:
brightness_preview = RUN_DIR / 'figures' / f'segmentation_brightness_s{BRIGHTNESS_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(brightness_preview)))


### DeepLabV3+: Gaussian blur evaluation


In [ ]:
GAUSSIAN_BLUR_SEVERITY = 1
if GAUSSIAN_BLUR_SEVERITY not in {1, 2, 3}:
    raise ValueError('GAUSSIAN_BLUR_SEVERITY must be 1, 2, or 3')
os.environ['GAUSSIAN_BLUR_SEVERITY'] = str(GAUSSIAN_BLUR_SEVERITY)
print('Gaussian blur severity:', GAUSSIAN_BLUR_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition gaussian_blur --severity "$GAUSSIAN_BLUR_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_gaussian_blur_s${GAUSSIAN_BLUR_SEVERITY}.log"


### DeepLabV3+: Gaussian blur segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition gaussian_blur --severity "$GAUSSIAN_BLUR_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_gaussian_blur_s${GAUSSIAN_BLUR_SEVERITY}.log"


In [ ]:
gaussian_blur_preview = RUN_DIR / 'figures' / f'segmentation_gaussian_blur_s{GAUSSIAN_BLUR_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(gaussian_blur_preview)))


### DeepLabV3+: Gaussian noise evaluation


In [ ]:
GAUSSIAN_NOISE_SEVERITY = 1
if GAUSSIAN_NOISE_SEVERITY not in {1, 2, 3}:
    raise ValueError('GAUSSIAN_NOISE_SEVERITY must be 1, 2, or 3')
os.environ['GAUSSIAN_NOISE_SEVERITY'] = str(GAUSSIAN_NOISE_SEVERITY)
print('Gaussian noise severity:', GAUSSIAN_NOISE_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition gaussian_noise --severity "$GAUSSIAN_NOISE_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_gaussian_noise_s${GAUSSIAN_NOISE_SEVERITY}.log"


### DeepLabV3+: Gaussian noise segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition gaussian_noise --severity "$GAUSSIAN_NOISE_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_gaussian_noise_s${GAUSSIAN_NOISE_SEVERITY}.log"


In [ ]:
gaussian_noise_preview = RUN_DIR / 'figures' / f'segmentation_gaussian_noise_s{GAUSSIAN_NOISE_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(gaussian_noise_preview)))


### DeepLabV3+: JPEG compression evaluation


In [ ]:
JPEG_COMPRESSION_SEVERITY = 1
if JPEG_COMPRESSION_SEVERITY not in {1, 2, 3}:
    raise ValueError('JPEG_COMPRESSION_SEVERITY must be 1, 2, or 3')
os.environ['JPEG_COMPRESSION_SEVERITY'] = str(JPEG_COMPRESSION_SEVERITY)
print('JPEG compression severity:', JPEG_COMPRESSION_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition jpeg_compression --severity "$JPEG_COMPRESSION_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_jpeg_compression_s${JPEG_COMPRESSION_SEVERITY}.log"


### DeepLabV3+: JPEG compression segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition jpeg_compression --severity "$JPEG_COMPRESSION_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_jpeg_compression_s${JPEG_COMPRESSION_SEVERITY}.log"


In [ ]:
jpeg_compression_preview = RUN_DIR / 'figures' / f'segmentation_jpeg_compression_s{JPEG_COMPRESSION_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(jpeg_compression_preview)))


### DeepLabV3+: Fog veil evaluation


In [ ]:
FOG_SEVERITY = 1
if FOG_SEVERITY not in {1, 2, 3}:
    raise ValueError('FOG_SEVERITY must be 1, 2, or 3')
os.environ['FOG_SEVERITY'] = str(FOG_SEVERITY)
print('Fog veil severity:', FOG_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition fog --severity "$FOG_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_fog_s${FOG_SEVERITY}.log"


### DeepLabV3+: Fog veil segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition fog --severity "$FOG_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_fog_s${FOG_SEVERITY}.log"


In [ ]:
fog_preview = RUN_DIR / 'figures' / f'segmentation_fog_s{FOG_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(fog_preview)))


### DeepLabV3+: batch preview selected image indices

Use this optional block after you have chosen several good validation image indices.


In [ ]:
PREVIEW_INDICES = [0, 1, 2]
if not PREVIEW_INDICES:
    raise ValueError('PREVIEW_INDICES must contain at least one image index')

BATCH_PREVIEW_SEVERITIES = {
    'darkness': globals().get('DARKNESS_SEVERITY', 1),
    'brightness': globals().get('BRIGHTNESS_SEVERITY', 1),
    'gaussian_blur': globals().get('GAUSSIAN_BLUR_SEVERITY', 1),
    'gaussian_noise': globals().get('GAUSSIAN_NOISE_SEVERITY', 1),
    'jpeg_compression': globals().get('JPEG_COMPRESSION_SEVERITY', 1),
    'fog': globals().get('FOG_SEVERITY', 1),
}
for condition, severity in BATCH_PREVIEW_SEVERITIES.items():
    if severity not in {1, 2, 3}:
        raise ValueError(f'{condition} severity must be 1, 2, or 3')

os.environ['PREVIEW_INDICES'] = ' '.join(str(index) for index in PREVIEW_INDICES)
os.environ['DARKNESS_SEVERITY'] = str(BATCH_PREVIEW_SEVERITIES['darkness'])
os.environ['BRIGHTNESS_SEVERITY'] = str(BATCH_PREVIEW_SEVERITIES['brightness'])
os.environ['GAUSSIAN_BLUR_SEVERITY'] = str(BATCH_PREVIEW_SEVERITIES['gaussian_blur'])
os.environ['GAUSSIAN_NOISE_SEVERITY'] = str(BATCH_PREVIEW_SEVERITIES['gaussian_noise'])
os.environ['JPEG_COMPRESSION_SEVERITY'] = str(BATCH_PREVIEW_SEVERITIES['jpeg_compression'])
os.environ['FOG_SEVERITY'] = str(BATCH_PREVIEW_SEVERITIES['fog'])

print('Batch preview indices:', PREVIEW_INDICES)
print('Batch preview severities:', BATCH_PREVIEW_SEVERITIES)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"

{
  for image_index in $PREVIEW_INDICES; do
    echo "Preview image index: ${image_index}"

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition clean

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition darkness --severity "$DARKNESS_SEVERITY"

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition brightness --severity "$BRIGHTNESS_SEVERITY"

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition gaussian_blur --severity "$GAUSSIAN_BLUR_SEVERITY"

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition gaussian_noise --severity "$GAUSSIAN_NOISE_SEVERITY"

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition jpeg_compression --severity "$JPEG_COMPRESSION_SEVERITY"

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition fog --severity "$FOG_SEVERITY"
  done
} 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_batch.log"


In [ ]:
batch_preview_files = []
for image_index in PREVIEW_INDICES:
    batch_preview_files.append(RUN_DIR / 'figures' / f'segmentation_clean_index_{image_index}.png')
    for condition, severity in BATCH_PREVIEW_SEVERITIES.items():
        batch_preview_files.append(
            RUN_DIR / 'figures' / f'segmentation_{condition}_s{severity}_index_{image_index}.png'
        )

for preview_path in batch_preview_files:
    print(preview_path.name)
    display(Image(filename=str(preview_path)))


### DeepLabV3+: saved results


In [ ]:
import pandas as pd

print('Results dir:', RUN_DIR)
print('Model dir:', MODEL_DIR)
display(pd.read_csv(RUN_DIR / 'metrics' / 'training_history.csv'))
display(pd.read_csv(RUN_DIR / 'metrics' / 'evaluation_results.csv'))

for plot_name in [
    'training_loss_curve.png',
    'dev_miou_curve.png',
    'dev_per_class_iou_curve.png',
]:
    plot_path = RUN_DIR / 'figures' / plot_name
    print(plot_path)
    display(Image(filename=str(plot_path)))


## PSPNet

Collapse this section when you work with another model.


### PSPNet: parameters


In [ ]:
PSPNET_SETTINGS = {
    'model': {
        'name': 'pspnet',
        'encoder_name': 'resnet34',
        'encoder_weights': 'imagenet',
    },
    'training': {
        'epochs': 8,
        'batch_size': 8,
        'num_workers': 2,
        'log_interval': 25,
        'learning_rate': 0.0003,
        'weight_decay': 0.0001,
        'device': 'auto',
        'mixed_precision': True,
    },
    'evaluation': {
        'batch_size': 8,
    },
}

prepare_model_run(
    selected_model='pspnet',
    run_name='pspnet_experiment_01',
    resume_training=False,
    continue_from_run=None,
    init_checkpoint='last',
    model_settings=PSPNET_SETTINGS,
    update_corruption_settings=False,
)


### PSPNet: train, resume, or continue


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
if [[ -n "$CONTINUE_FROM_RUN" ]]; then
  python -u scripts/train_model.py --config "$RUN_CONFIG" --continue-from-run "$CONTINUE_FROM_RUN" --init-checkpoint "$INIT_CHECKPOINT" 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
elif [[ "$RESUME_TRAINING" == "1" ]]; then
  python -u scripts/train_model.py --config "$RUN_CONFIG" --resume 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
else
  python -u scripts/train_model.py --config "$RUN_CONFIG" 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
fi


In [ ]:
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


### PSPNet: preview image index

Set one official validation image index for all manual preview cells below.


In [ ]:
IMAGE_INDEX = 0
os.environ['IMAGE_INDEX'] = str(IMAGE_INDEX)
print('Preview image index:', IMAGE_INDEX)


### PSPNet: clean segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition clean 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_clean.log"


In [ ]:
clean_preview = RUN_DIR / 'figures' / f'segmentation_clean_index_{IMAGE_INDEX}.png'
display(Image(filename=str(clean_preview)))


### PSPNet: clean evaluation


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition clean 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_clean.log"


### PSPNet: Darkness evaluation


In [ ]:
DARKNESS_SEVERITY = 1
if DARKNESS_SEVERITY not in {1, 2, 3}:
    raise ValueError('DARKNESS_SEVERITY must be 1, 2, or 3')
os.environ['DARKNESS_SEVERITY'] = str(DARKNESS_SEVERITY)
print('Darkness severity:', DARKNESS_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition darkness --severity "$DARKNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_darkness_s${DARKNESS_SEVERITY}.log"


### PSPNet: Darkness segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition darkness --severity "$DARKNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_darkness_s${DARKNESS_SEVERITY}.log"


In [ ]:
darkness_preview = RUN_DIR / 'figures' / f'segmentation_darkness_s{DARKNESS_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(darkness_preview)))


### PSPNet: Increased brightness evaluation


In [ ]:
BRIGHTNESS_SEVERITY = 1
if BRIGHTNESS_SEVERITY not in {1, 2, 3}:
    raise ValueError('BRIGHTNESS_SEVERITY must be 1, 2, or 3')
os.environ['BRIGHTNESS_SEVERITY'] = str(BRIGHTNESS_SEVERITY)
print('Increased brightness severity:', BRIGHTNESS_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition brightness --severity "$BRIGHTNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_brightness_s${BRIGHTNESS_SEVERITY}.log"


### PSPNet: Increased brightness segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition brightness --severity "$BRIGHTNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_brightness_s${BRIGHTNESS_SEVERITY}.log"


In [ ]:
brightness_preview = RUN_DIR / 'figures' / f'segmentation_brightness_s{BRIGHTNESS_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(brightness_preview)))


### PSPNet: Gaussian blur evaluation


In [ ]:
GAUSSIAN_BLUR_SEVERITY = 1
if GAUSSIAN_BLUR_SEVERITY not in {1, 2, 3}:
    raise ValueError('GAUSSIAN_BLUR_SEVERITY must be 1, 2, or 3')
os.environ['GAUSSIAN_BLUR_SEVERITY'] = str(GAUSSIAN_BLUR_SEVERITY)
print('Gaussian blur severity:', GAUSSIAN_BLUR_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition gaussian_blur --severity "$GAUSSIAN_BLUR_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_gaussian_blur_s${GAUSSIAN_BLUR_SEVERITY}.log"


### PSPNet: Gaussian blur segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition gaussian_blur --severity "$GAUSSIAN_BLUR_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_gaussian_blur_s${GAUSSIAN_BLUR_SEVERITY}.log"


In [ ]:
gaussian_blur_preview = RUN_DIR / 'figures' / f'segmentation_gaussian_blur_s{GAUSSIAN_BLUR_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(gaussian_blur_preview)))


### PSPNet: Gaussian noise evaluation


In [ ]:
GAUSSIAN_NOISE_SEVERITY = 1
if GAUSSIAN_NOISE_SEVERITY not in {1, 2, 3}:
    raise ValueError('GAUSSIAN_NOISE_SEVERITY must be 1, 2, or 3')
os.environ['GAUSSIAN_NOISE_SEVERITY'] = str(GAUSSIAN_NOISE_SEVERITY)
print('Gaussian noise severity:', GAUSSIAN_NOISE_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition gaussian_noise --severity "$GAUSSIAN_NOISE_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_gaussian_noise_s${GAUSSIAN_NOISE_SEVERITY}.log"


### PSPNet: Gaussian noise segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition gaussian_noise --severity "$GAUSSIAN_NOISE_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_gaussian_noise_s${GAUSSIAN_NOISE_SEVERITY}.log"


In [ ]:
gaussian_noise_preview = RUN_DIR / 'figures' / f'segmentation_gaussian_noise_s{GAUSSIAN_NOISE_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(gaussian_noise_preview)))


### PSPNet: JPEG compression evaluation


In [ ]:
JPEG_COMPRESSION_SEVERITY = 1
if JPEG_COMPRESSION_SEVERITY not in {1, 2, 3}:
    raise ValueError('JPEG_COMPRESSION_SEVERITY must be 1, 2, or 3')
os.environ['JPEG_COMPRESSION_SEVERITY'] = str(JPEG_COMPRESSION_SEVERITY)
print('JPEG compression severity:', JPEG_COMPRESSION_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition jpeg_compression --severity "$JPEG_COMPRESSION_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_jpeg_compression_s${JPEG_COMPRESSION_SEVERITY}.log"


### PSPNet: JPEG compression segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition jpeg_compression --severity "$JPEG_COMPRESSION_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_jpeg_compression_s${JPEG_COMPRESSION_SEVERITY}.log"


In [ ]:
jpeg_compression_preview = RUN_DIR / 'figures' / f'segmentation_jpeg_compression_s{JPEG_COMPRESSION_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(jpeg_compression_preview)))


### PSPNet: Fog veil evaluation


In [ ]:
FOG_SEVERITY = 1
if FOG_SEVERITY not in {1, 2, 3}:
    raise ValueError('FOG_SEVERITY must be 1, 2, or 3')
os.environ['FOG_SEVERITY'] = str(FOG_SEVERITY)
print('Fog veil severity:', FOG_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition fog --severity "$FOG_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_fog_s${FOG_SEVERITY}.log"


### PSPNet: Fog veil segmentation preview


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$IMAGE_INDEX" --condition fog --severity "$FOG_SEVERITY" 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_fog_s${FOG_SEVERITY}.log"


In [ ]:
fog_preview = RUN_DIR / 'figures' / f'segmentation_fog_s{FOG_SEVERITY}_index_{IMAGE_INDEX}.png'
display(Image(filename=str(fog_preview)))


### PSPNet: batch preview selected image indices

Use this optional block after you have chosen several good validation image indices.


In [ ]:
PREVIEW_INDICES = [0, 1, 2]
if not PREVIEW_INDICES:
    raise ValueError('PREVIEW_INDICES must contain at least one image index')

BATCH_PREVIEW_SEVERITIES = {
    'darkness': globals().get('DARKNESS_SEVERITY', 1),
    'brightness': globals().get('BRIGHTNESS_SEVERITY', 1),
    'gaussian_blur': globals().get('GAUSSIAN_BLUR_SEVERITY', 1),
    'gaussian_noise': globals().get('GAUSSIAN_NOISE_SEVERITY', 1),
    'jpeg_compression': globals().get('JPEG_COMPRESSION_SEVERITY', 1),
    'fog': globals().get('FOG_SEVERITY', 1),
}
for condition, severity in BATCH_PREVIEW_SEVERITIES.items():
    if severity not in {1, 2, 3}:
        raise ValueError(f'{condition} severity must be 1, 2, or 3')

os.environ['PREVIEW_INDICES'] = ' '.join(str(index) for index in PREVIEW_INDICES)
os.environ['DARKNESS_SEVERITY'] = str(BATCH_PREVIEW_SEVERITIES['darkness'])
os.environ['BRIGHTNESS_SEVERITY'] = str(BATCH_PREVIEW_SEVERITIES['brightness'])
os.environ['GAUSSIAN_BLUR_SEVERITY'] = str(BATCH_PREVIEW_SEVERITIES['gaussian_blur'])
os.environ['GAUSSIAN_NOISE_SEVERITY'] = str(BATCH_PREVIEW_SEVERITIES['gaussian_noise'])
os.environ['JPEG_COMPRESSION_SEVERITY'] = str(BATCH_PREVIEW_SEVERITIES['jpeg_compression'])
os.environ['FOG_SEVERITY'] = str(BATCH_PREVIEW_SEVERITIES['fog'])

print('Batch preview indices:', PREVIEW_INDICES)
print('Batch preview severities:', BATCH_PREVIEW_SEVERITIES)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"

{
  for image_index in $PREVIEW_INDICES; do
    echo "Preview image index: ${image_index}"

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition clean

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition darkness --severity "$DARKNESS_SEVERITY"

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition brightness --severity "$BRIGHTNESS_SEVERITY"

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition gaussian_blur --severity "$GAUSSIAN_BLUR_SEVERITY"

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition gaussian_noise --severity "$GAUSSIAN_NOISE_SEVERITY"

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition jpeg_compression --severity "$JPEG_COMPRESSION_SEVERITY"

    python scripts/visualize_checkpoint.py --config "$RUN_CONFIG" --index "$image_index" --condition fog --severity "$FOG_SEVERITY"
  done
} 2>&1 | tee -a "$LOG_DIR/preview_${RUN_NAME}_batch.log"


In [ ]:
batch_preview_files = []
for image_index in PREVIEW_INDICES:
    batch_preview_files.append(RUN_DIR / 'figures' / f'segmentation_clean_index_{image_index}.png')
    for condition, severity in BATCH_PREVIEW_SEVERITIES.items():
        batch_preview_files.append(
            RUN_DIR / 'figures' / f'segmentation_{condition}_s{severity}_index_{image_index}.png'
        )

for preview_path in batch_preview_files:
    print(preview_path.name)
    display(Image(filename=str(preview_path)))


### PSPNet: saved results


In [ ]:
import pandas as pd

print('Results dir:', RUN_DIR)
print('Model dir:', MODEL_DIR)
display(pd.read_csv(RUN_DIR / 'metrics' / 'training_history.csv'))
display(pd.read_csv(RUN_DIR / 'metrics' / 'evaluation_results.csv'))

for plot_name in [
    'training_loss_curve.png',
    'dev_miou_curve.png',
    'dev_per_class_iou_curve.png',
]:
    plot_path = RUN_DIR / 'figures' / plot_name
    print(plot_path)
    display(Image(filename=str(plot_path)))


## 16. MLflow UI

Use these small cells in order: start the server, check it, open a browser link or iframe, and stop it when finished.


In [ ]:
import os
import subprocess
import time
from pathlib import Path
from google.colab import output
from IPython.display import display, HTML

DRIVE_ROOT = Path('/content/drive/MyDrive/cityscapes_robustness')
MLFLOW_DB = DRIVE_ROOT / 'mlflow.db'
MLFLOW_ARTIFACT_DIR = DRIVE_ROOT / 'mlartifacts'
LOG_DIR = DRIVE_ROOT / 'logs'

LOG_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

backend_store_uri = f'sqlite:///{MLFLOW_DB}'
artifact_root_uri = MLFLOW_ARTIFACT_DIR.as_uri()

os.environ['MLFLOW_TRACKING_URI'] = backend_store_uri
os.environ['MLFLOW_ARTIFACT_ROOT'] = artifact_root_uri

subprocess.run("pkill -f 'mlflow server' || true", shell=True, check=False)

mlflow_log = LOG_DIR / 'mlflow_ui_server.log'
cmd = [
    'mlflow',
    'server',
    '--backend-store-uri',
    backend_store_uri,
    '--default-artifact-root',
    artifact_root_uri,
    '--host',
    '0.0.0.0',
    '--port',
    '5000',
    '--allowed-hosts',
    '*',
    '--cors-allowed-origins',
    '*',
    '--x-frame-options',
    'NONE',
    '--workers',
    '1',
]

print('Starting MLflow UI:')
print(' '.join(cmd))
print('Server log:', mlflow_log)

log_file = open(mlflow_log, 'a', encoding='utf-8')
mlflow_process = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)
time.sleep(5)

if mlflow_process.poll() is not None:
    print('MLflow server stopped with an error. Last log lines:')
    print(mlflow_log.read_text(encoding='utf-8')[-4000:])
    raise RuntimeError('MLflow server did not start')

url = output.eval_js('google.colab.kernel.proxyPort(5000)')
print('MLflow server PID:', mlflow_process.pid)
print('MLflow UI URL:', url)
display(HTML(f'<a href="{url}" target="_blank" style="font-size:18px">Open MLflow UI in a new tab</a>'))
output.serve_kernel_port_as_iframe(5000, height=800)


### Check MLflow UI connection


In [ ]:
!curl -I http://127.0.0.1:5000


### Reopen MLflow UI link or iframe


In [ ]:
from google.colab import output
from IPython.display import display, HTML

url = output.eval_js('google.colab.kernel.proxyPort(5000)')
print(url)
display(HTML(f'<a href="{url}" target="_blank" style="font-size:18px">Open MLflow UI in a new tab</a>'))
output.serve_kernel_port_as_iframe(5000, height=800)


### Stop MLflow UI server


In [ ]:
!pkill -f "mlflow server" || true


## Resume and continue training

Resume after a runtime interruption:

1. Run sections 1-8 again.
2. In section 9, set the old `RUN_NAME`, `RESUME_TRAINING = True`, `CONTINUE_FROM_RUN = None`.
3. Run section 10. Existing `run_config.yaml` will not be overwritten.

Continue training in a new run:

1. In section 9, set a new `RUN_NAME`.
2. Set `CONTINUE_FROM_RUN = 'old_run_name'` and `RESUME_TRAINING = False`.
3. Set `training.epochs` in this model section to the number of new epochs to add.
4. Run section 10. The new `run_config.yaml` will store the source run, source model folder, source checkpoint, `additional_epochs`, `initial_checkpoint_epoch`, and total `training.epochs`.

Folder reminder:

- `RUN_DIR = DRIVE_ROOT / 'runs' / RUN_NAME` contains results only: metrics, CSV, PNG, config, MLflow run id.
- `MODEL_DIR = DRIVE_ROOT / 'models' / RUN_NAME` contains checkpoints only: `best.pt`, `last.pt`.

Clean evaluation and any selected corruption evaluation can be run repeatedly: every evaluation is saved separately.
